# Raw Data Exploration

Interactive exploration of the `Solenopsisbot/real-slop` dataset.  
Focus: browsing individual rows and investigating deduplication.

In [1]:
import polars as pl
import hashlib
from IPython.display import display, HTML, Markdown
from pathlib import Path
import textwrap
import json

PARQUET = Path("../data/raw/real_slop.parquet")
df = pl.read_parquet(PARQUET)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns}")
print(f"\nMemory usage: {df.estimated_size('mb'):.1f} MB")

Loaded 155,623 rows, 6 columns
Columns: ['messages', 'response', 'reasoning', 'model', 'tools', 'tool_calls']

Memory usage: 17664.7 MB


## 1. Schema & Basic Stats

In [2]:
# Column types, null counts, and sample values
for col in df.columns:
    non_null = df[col].drop_nulls().len()
    null_pct = (1 - non_null / len(df)) * 100
    print(f"{col:15s}  dtype={str(df[col].dtype):8s}  non-null={non_null:>7,}  null={null_pct:5.1f}%")

messages         dtype=String    non-null=155,623  null=  0.0%
response         dtype=String    non-null=149,477  null=  3.9%
reasoning        dtype=String    non-null= 11,366  null= 92.7%
model            dtype=String    non-null=155,623  null=  0.0%
tools            dtype=String    non-null=  4,537  null= 97.1%
tool_calls       dtype=String    non-null=  9,506  null= 93.9%


## 2. Browse Individual Rows

Helper to display a single row with readable text wrapping.

In [3]:
def show_row(idx: int, max_chars: int = 2000) -> None:
    """Display a single row with readable formatting.
    
    Args:
        idx: Row index (0-based)
        max_chars: Max characters to show per field (0 = no limit)
    """
    row = df.row(idx, named=True)
    
    def truncate(text, limit):
        if text is None:
            return "<null>"
        text = str(text)
        if limit and len(text) > limit:
            return text[:limit] + f"\n... [truncated, {len(text):,} chars total]"
        return text
    
    print(f"{'='*80}")
    print(f"ROW {idx:,} of {len(df):,}")
    print(f"{'='*80}")
    print(f"\nModel: {row['model']}")
    
    # Parse messages to show conversation structure
    print(f"\n{'─'*40} MESSAGES {'─'*40}")
    try:
        msgs = json.loads(row['messages'])
        if isinstance(msgs, list):
            for i, msg in enumerate(msgs):
                role = msg.get('role', '?')
                content = msg.get('content', '')
                if isinstance(content, list):
                    parts = [b.get('text', '') for b in content if isinstance(b, dict) and b.get('type') == 'text']
                    content = ' '.join(parts)
                content = str(content)
                print(f"\n  [{i}] {role.upper()}:")
                print(textwrap.indent(truncate(content, max_chars), '      '))
        else:
            print(truncate(row['messages'], max_chars))
    except (json.JSONDecodeError, TypeError):
        print(truncate(row['messages'], max_chars))
    
    print(f"\n{'─'*40} REASONING {'─'*39}")
    print(truncate(row['reasoning'], max_chars))
    
    print(f"\n{'─'*40} RESPONSE {'─'*40}")
    print(truncate(row['response'], max_chars))
    
    if row['tools'] is not None:
        print(f"\n{'─'*40} TOOLS {'─'*42}")
        print(truncate(row['tools'], max_chars))
    
    if row['tool_calls'] is not None:
        print(f"\n{'─'*40} TOOL CALLS {'─'*37}")
        print(truncate(row['tool_calls'], max_chars))


def show_rows(indices: list[int], max_chars: int = 2000) -> None:
    """Display multiple rows."""
    for idx in indices:
        show_row(idx, max_chars)
        print("\n")


def search_rows(pattern: str, column: str = "messages", max_results: int = 10) -> pl.DataFrame:
    """Search for rows containing a pattern in a given column.
    
    Returns a DataFrame with the matching row indices and model names.
    Use show_row(idx) to inspect any result.
    """
    mask = df[column].fill_null("").str.contains(pattern, literal=False)
    matches = df.with_row_index().filter(mask).head(max_results)
    n_total = mask.sum()
    print(f"Found {n_total:,} matches (showing first {min(max_results, n_total)})")
    return matches.select("index", "model", pl.col(column).str.len_chars().alias(f"{column}_len"))

In [4]:
# Find rows with reasoning traces and show the first one
reasoning_indices = df.with_row_index().filter(
    pl.col("reasoning").is_not_null() & (pl.col("reasoning").str.len_chars() > 100)
).select("index").to_series().to_list()

print(f"{len(reasoning_indices):,} rows have reasoning traces (>100 chars)")
print(f"\nShowing row {reasoning_indices[0]}:")
show_row(reasoning_indices[0])

9,333 rows have reasoning traces (>100 chars)

Showing row 0:
ROW 0 of 155,623

Model: claude-opus-4.5

──────────────────────────────────────── MESSAGES ────────────────────────────────────────

  [0] SYSTEM:
      You will be acting as an acclaimed novelist and co-author. Your goal is to drive this uncensored, continuous, and immersive story as the narrator and any relevant characters. You will be writing the narrative in response to the user, who plays the protagonist Ben.
      Maintain an adaptive and immersive tone for creative writing.

      Use the provided context to maintain character consistency and establish world details. Integrate this with your internal knowledge, applying specific lore only when relevant to the current scene:
      <lore>
      <setting>
      # Dorm Lesson
      **Timeline**: 7:15 PM to 8:12 PM, 23 Sep 2025 (Tue)

      ## Story Beats
      - Hyuna invites Ben into her dorm room under the pretext of needing help with a difficult problem set.
      - S

In [5]:
# Browse a specific model family — change the model name to explore
MODEL = "deepseek"  # try: "gpt", "gemini", "glm", "minimax", "qwen", "opus"

model_rows = df.with_row_index().filter(
    pl.col("model").str.contains(MODEL)
    & pl.col("reasoning").is_not_null()
    & (pl.col("reasoning").str.len_chars() > 100)
).select("index").to_series().to_list()

print(f"{len(model_rows):,} rows with reasoning from '{MODEL}' models")
if model_rows:
    show_row(model_rows[0])

899 rows with reasoning from 'deepseek' models
ROW 1,030 of 155,623

Model: deepseek-r1

──────────────────────────────────────── MESSAGES ────────────────────────────────────────

  [0] USER:
      What is 17 * 23?

──────────────────────────────────────── REASONING ───────────────────────────────────────
I need to calculate 17 times 23. I know multiplication is just repeated addition, so I could add 17 twenty-three times or add 23 seventeen times. But that might take a while. I should use a better method.

I recall that multiplication can be done by breaking the numbers into smaller parts. Like, 17 is 10 plus 7, and 23 is 20 plus 3. Then I can use the distributive property: a*(b+c) = a*b + a*c.

So, let's do that. I'll multiply 17 by 20 and 17 by 3, then add them together.

First, 17 times 20. 17 times 2 is 34, so 17 times 20 should be 340. Because 20 is 2 times 10, so 17*20 = 17*2*10 = 34*10 = 340.

Now, 17 times 3. That's 17*3 = 51.

So, now I add them: 340 + 51. 340 + 50 is 390, p

In [6]:
# Search for rows mentioning a specific topic in messages
# Change the pattern to search for anything
results = search_rows("machine learning", column="messages", max_results=5)
results

Found 1,577 matches (showing first 5)


index,model,messages_len
u32,str,u32
2072,"""claude-opus-4.5""",98747
2080,"""claude-opus-4.5""",60500
4557,"""claude-sonnet-4.5""",2309
4567,"""claude-sonnet-4.5""",2436
4581,"""claude-sonnet-4.5""",2288


## 3. Deduplication Analysis

Check for duplicates at multiple levels:
1. **Exact row duplicates** — identical across all columns
2. **Messages duplicates** — same user messages (different responses)
3. **Response duplicates** — same response text
4. **Reasoning duplicates** — same reasoning trace
5. **Near-duplicates** — messages that differ only trivially

In [7]:
# 3.1 Exact row duplicates (all columns identical)
n_total = len(df)
n_unique = df.unique().shape[0]
n_exact_dupes = n_total - n_unique

print(f"Total rows:           {n_total:>10,}")
print(f"Unique rows:          {n_unique:>10,}")
print(f"Exact duplicate rows: {n_exact_dupes:>10,}  ({n_exact_dupes/n_total*100:.2f}%)")

Total rows:              155,623
Unique rows:             153,085
Exact duplicate rows:      2,538  (1.63%)


In [8]:
# 3.2 Column-level duplicate analysis
print(f"{'Column':<15} {'Total non-null':>15} {'Unique':>15} {'Duplicated':>15} {'Dup %':>10}")
print("─" * 72)

for col in ["messages", "response", "reasoning"]:
    non_null = df[col].drop_nulls()
    n = len(non_null)
    n_uniq = non_null.n_unique()
    n_dup = n - n_uniq
    pct = n_dup / n * 100 if n > 0 else 0
    print(f"{col:<15} {n:>15,} {n_uniq:>15,} {n_dup:>15,} {pct:>9.2f}%")

Column           Total non-null          Unique      Duplicated      Dup %
────────────────────────────────────────────────────────────────────────
messages                155,623         133,092          22,531     14.48%
response                149,477         138,643          10,834      7.25%
reasoning                11,366           9,437           1,929     16.97%


In [9]:
# 3.3 Messages sent to multiple models (same prompt, different model)
# These are NOT "bad" duplicates — they represent the same user query sent to different models
msg_model = df.select("messages", "model").drop_nulls()
msg_groups = msg_model.group_by("messages").agg(
    pl.col("model").n_unique().alias("n_models"),
    pl.len().alias("n_rows"),
    pl.col("model").alias("models"),
)

multi_model = msg_groups.filter(pl.col("n_models") > 1)
print(f"Unique message texts: {len(msg_groups):,}")
print(f"Messages sent to >1 model: {len(multi_model):,}")
print(f"Rows involved: {multi_model['n_rows'].sum():,}")

print(f"\nDistribution of models-per-message (for multi-model messages):")
print(multi_model["n_models"].describe())

# Show distribution
print(f"\nTop model counts:")
print(multi_model.group_by("n_models").len().sort("n_models"))

Unique message texts: 133,092
Messages sent to >1 model: 1,388
Rows involved: 10,001

Distribution of models-per-message (for multi-model messages):
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 1388.0   │
│ null_count ┆ 0.0      │
│ mean       ┆ 3.151297 │
│ std        ┆ 5.85047  │
│ min        ┆ 2.0      │
│ 25%        ┆ 2.0      │
│ 50%        ┆ 2.0      │
│ 75%        ┆ 2.0      │
│ max        ┆ 86.0     │
└────────────┴──────────┘

Top model counts:
shape: (31, 2)
┌──────────┬──────┐
│ n_models ┆ len  │
│ ---      ┆ ---  │
│ u32      ┆ u32  │
╞══════════╪══════╡
│ 2        ┆ 1083 │
│ 3        ┆ 141  │
│ 4        ┆ 57   │
│ 5        ┆ 29   │
│ 6        ┆ 13   │
│ …        ┆ …    │
│ 69       ┆ 1    │
│ 70       ┆ 1    │
│ 73       ┆ 1    │
│ 78       ┆ 1    │
│ 86       ┆ 1    │
└──────────┴──────┘


In [10]:
# 3.4 True duplicates: same messages AND same model (re-runs or actual duplicates)
msg_model_groups = df.group_by("messages", "model").agg(
    pl.len().alias("n_rows"),
    pl.col("response").n_unique().alias("n_unique_responses"),
)

same_model_dupes = msg_model_groups.filter(pl.col("n_rows") > 1)
print(f"(messages, model) pairs appearing >1 time: {len(same_model_dupes):,}")
print(f"Total rows in duplicated pairs: {same_model_dupes['n_rows'].sum():,}")

# Of these, how many have different responses? (re-generations vs exact copies)
with_diff_responses = same_model_dupes.filter(pl.col("n_unique_responses") > 1)
with_same_responses = same_model_dupes.filter(pl.col("n_unique_responses") == 1)
print(f"\n  With different responses (re-generations): {len(with_diff_responses):,}")
print(f"  With identical responses (exact copies):   {len(with_same_responses):,}")

(messages, model) pairs appearing >1 time: 7,770
Total rows in duplicated pairs: 27,315

  With different responses (re-generations): 7,514
  With identical responses (exact copies):   256


In [11]:
# 3.5 Inspect examples of duplicates
# Find a duplicated (messages, model) pair and show both rows
if len(same_model_dupes) > 0:
    # Pick one with >1 duplicate
    sample_dupe = same_model_dupes.sort("n_rows", descending=True).head(1)
    dupe_msg = sample_dupe["messages"][0]
    dupe_model = sample_dupe["model"][0]
    
    dupe_rows = df.with_row_index().filter(
        (pl.col("messages") == dupe_msg) & (pl.col("model") == dupe_model)
    )
    
    print(f"Example: {sample_dupe['n_rows'][0]} rows with same messages + model '{dupe_model}'")
    print(f"Unique responses among them: {dupe_rows['response'].n_unique()}")
    print(f"Unique reasoning among them: {dupe_rows['reasoning'].drop_nulls().n_unique()}")
    
    indices = dupe_rows["index"].to_list()
    print(f"\nRow indices: {indices[:5]}{'...' if len(indices) > 5 else ''}")
    print(f"\nShowing first two:")
    show_rows(indices[:2], max_chars=500)
else:
    print("No same-model duplicates found.")

Example: 617 rows with same messages + model 'claude-opus-4-5-20251101'
Unique responses among them: 8
Unique reasoning among them: 95

Row indices: [808, 1750, 1751, 1752, 1753]...

Showing first two:
ROW 808 of 155,623

Model: claude-opus-4-5-20251101

──────────────────────────────────────── MESSAGES ────────────────────────────────────────

  [0] USER:
      Say hi

──────────────────────────────────────── REASONING ───────────────────────────────────────
<null>

──────────────────────────────────────── RESPONSE ────────────────────────────────────────
Hi there! How are you doing today? Is there something I can help you with?


ROW 1,750 of 155,623

Model: claude-opus-4-5-20251101

──────────────────────────────────────── MESSAGES ────────────────────────────────────────

  [0] USER:
      Say hi

──────────────────────────────────────── REASONING ───────────────────────────────────────
<null>

──────────────────────────────────────── RESPONSE ──────────────────────────────────────

In [12]:
# 3.6 Reasoning trace duplicates (important for our analysis)
reasoning_df = df.filter(pl.col("reasoning").is_not_null()).select("reasoning", "model", "messages")

n_reasoning = len(reasoning_df)
n_unique_reasoning = reasoning_df["reasoning"].n_unique()
n_dup_reasoning = n_reasoning - n_unique_reasoning

print(f"Rows with reasoning:     {n_reasoning:>10,}")
print(f"Unique reasoning texts:  {n_unique_reasoning:>10,}")
print(f"Duplicate reasoning:     {n_dup_reasoning:>10,}  ({n_dup_reasoning/n_reasoning*100:.2f}%)")

# Are duplicated reasoning traces from the same prompt or different prompts?
reasoning_groups = reasoning_df.group_by("reasoning").agg(
    pl.len().alias("n_rows"),
    pl.col("messages").n_unique().alias("n_unique_messages"),
    pl.col("model").n_unique().alias("n_unique_models"),
).filter(pl.col("n_rows") > 1)

print(f"\nDuplicated reasoning groups: {len(reasoning_groups):,}")
if len(reasoning_groups) > 0:
    print(f"  Same message & same reasoning: {reasoning_groups.filter(pl.col('n_unique_messages') == 1).shape[0]:,}")
    print(f"  Different message but same reasoning: {reasoning_groups.filter(pl.col('n_unique_messages') > 1).shape[0]:,}")

Rows with reasoning:         11,366
Unique reasoning texts:       9,437
Duplicate reasoning:          1,929  (16.97%)

Duplicated reasoning groups: 127
  Same message & same reasoning: 117
  Different message but same reasoning: 10


In [13]:
# 3.7 Near-duplicate detection via content hashing
# Hash the first 500 chars of messages to find near-duplicates
# (messages that start the same but may differ at the end)

near_dupe_df = df.with_columns(
    pl.col("messages").fill_null("").str.slice(0, 500).alias("msg_prefix")
)

prefix_groups = near_dupe_df.group_by("msg_prefix").agg(
    pl.len().alias("n_rows"),
    pl.col("messages").n_unique().alias("n_unique_full_msgs"),
)

near_dupes = prefix_groups.filter(
    (pl.col("n_rows") > 1) & (pl.col("n_unique_full_msgs") > 1)
)

print(f"Message prefixes (500 chars) with multiple rows: {prefix_groups.filter(pl.col('n_rows') > 1).shape[0]:,}")
print(f"Of those, with different full messages (near-dupes): {len(near_dupes):,}")
print(f"Total rows in near-duplicate groups: {near_dupes['n_rows'].sum() if len(near_dupes) > 0 else 0:,}")

Message prefixes (500 chars) with multiple rows: 4,901
Of those, with different full messages (near-dupes): 3,344
Total rows in near-duplicate groups: 130,331


In [14]:
# 3.8 Deduplication summary & recommendation
print("DEDUPLICATION SUMMARY")
print("=" * 60)
print(f"\nTotal rows:                      {n_total:>10,}")
print(f"Exact row duplicates:            {n_exact_dupes:>10,}  ({n_exact_dupes/n_total*100:.2f}%)")
print(f"Same (messages,model) >1 time:   {len(same_model_dupes):>10,}")
print(f"  - with identical responses:     {len(with_same_responses):>10,}  (exact copies)")
print(f"  - with different responses:     {len(with_diff_responses):>10,}  (re-generations)")
print(f"Reasoning duplicates:            {n_dup_reasoning:>10,}  ({n_dup_reasoning/n_reasoning*100:.2f}% of reasoning rows)")

print(f"\n{'─'*60}")
print("\nNotes:")
print("- Same messages sent to different models are NOT duplicates")
print("  (they're the cross-model comparisons in the dataset)")
print("- Same messages + same model with different responses are")
print("  re-generations (potentially useful for variance analysis)")
print("- Exact copies (same msg + model + response) should be deduped")

DEDUPLICATION SUMMARY

Total rows:                         155,623
Exact row duplicates:                 2,538  (1.63%)
Same (messages,model) >1 time:        7,770
  - with identical responses:            256  (exact copies)
  - with different responses:          7,514  (re-generations)
Reasoning duplicates:                 1,929  (16.97% of reasoning rows)

────────────────────────────────────────────────────────────

Notes:
- Same messages sent to different models are NOT duplicates
  (they're the cross-model comparisons in the dataset)
- Same messages + same model with different responses are
  re-generations (potentially useful for variance analysis)
- Exact copies (same msg + model + response) should be deduped


In [15]:
# 3.9 Impact on our analysis subset (reasoning rows only)
reasoning_only = df.filter(pl.col("reasoning").is_not_null())
n_r = len(reasoning_only)

reasoning_unique = reasoning_only.unique(subset=["messages", "model", "response"])
n_r_unique = len(reasoning_unique)

print(f"Reasoning rows (total):             {n_r:>8,}")
print(f"Reasoning rows (after dedup):       {n_r_unique:>8,}")
print(f"Duplicates removed:                 {n_r - n_r_unique:>8,}  ({(n_r - n_r_unique)/n_r*100:.2f}%)")

# Per model family (inline to avoid import path issues)
import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.preprocessing import extract_model_family

print(f"\nPer model family:")
reasoning_with_fam = reasoning_only.with_columns(
    extract_model_family(pl.col("model")).alias("model_family")
)
reasoning_deduped = reasoning_with_fam.unique(subset=["messages", "model", "response"])

before = reasoning_with_fam.group_by("model_family").len().sort("model_family")
after = reasoning_deduped.group_by("model_family").len().sort("model_family")

comparison = before.join(after, on="model_family", suffix="_deduped")
comparison = comparison.with_columns(
    (pl.col("len") - pl.col("len_deduped")).alias("removed"),
    ((pl.col("len") - pl.col("len_deduped")) / pl.col("len") * 100).round(1).alias("removed_pct"),
)
print(comparison.sort("len", descending=True))

Reasoning rows (total):               11,366
Reasoning rows (after dedup):          9,923
Duplicates removed:                    1,443  (12.70%)

Per model family:
shape: (10, 5)
┌───────────────┬──────┬─────────────┬─────────┬─────────────┐
│ model_family  ┆ len  ┆ len_deduped ┆ removed ┆ removed_pct │
│ ---           ┆ ---  ┆ ---         ┆ ---     ┆ ---         │
│ str           ┆ u32  ┆ u32         ┆ u32     ┆ f64         │
╞═══════════════╪══════╪═════════════╪═════════╪═════════════╡
│ glm           ┆ 3252 ┆ 3101        ┆ 151     ┆ 4.6         │
│ claude-opus   ┆ 2923 ┆ 1898        ┆ 1025    ┆ 35.1        │
│ gemini        ┆ 1596 ┆ 1582        ┆ 14      ┆ 0.9         │
│ minimax       ┆ 1432 ┆ 1423        ┆ 9       ┆ 0.6         │
│ deepseek      ┆ 933  ┆ 887         ┆ 46      ┆ 4.9         │
│ gpt           ┆ 659  ┆ 654         ┆ 5       ┆ 0.8         │
│ other         ┆ 394  ┆ 208         ┆ 186     ┆ 47.2        │
│ claude-sonnet ┆ 91   ┆ 91          ┆ 0       ┆ 0.0         │
│ 

## 4. Quick-Reference: Browse Any Row

Use these helpers to explore:

```python
# Show a specific row by index
show_row(42)

# Show full text (no truncation)
show_row(42, max_chars=0)

# Search for rows containing a pattern
results = search_rows("python", column="messages")
show_row(results["index"][0])  # show first match

# Search in reasoning traces
results = search_rows("I'm not sure", column="reasoning")

# Browse a specific model
show_row(reasoning_indices[100])  # 100th reasoning row
```

In [ ]:
# Sandbox cell — use this for ad-hoc exploration
# show_row(0)
# search_rows("your pattern here", column="reasoning")

In [21]:
show_row(11360, max_chars=0)

ROW 11,360 of 155,623

Model: claude-3-7-sonnet-20250219

──────────────────────────────────────── MESSAGES ────────────────────────────────────────

  [0] SYSTEM:
      Kimiko, you have gone through rigorous training and study past 5 years, double major in both STEM (math, practical physics) and liberal art (history, literature, rhetoric). All those rigorous and arduous study about culture, history, literature classics, physics finally bear fruit that you land into the WHALE publishing house, a rising publisher specialized in fiction story.

      For your first job, your task you should start working with existing material before creating everything your own. Those will be present shortly. For now, let say you will conduct business under the code name/alias DIPSY. DIPSY role/privilege := author.


      `Now hold on a minute, my major and education have nothing to do with literature study or fictional writing are you sure there is no misunderstanding`. 

      `I fit the bill?` 
    